# Laboratorio 3: Spezzare, cercare, misurare

---

# Lab 3: Split, search, measure

**Come si usa.** Esegui le celle dall'alto in basso (Maiusc+Invio). Serve un account Google gratuito
per usare Colab. Il testo completo del laboratorio e' sul sito: https://corso-rag.giuseppebosi.com/moduli/03-testo-cercabile/LAB/

---

**How to use it.** Run the cells from top to bottom (Shift+Enter). Colab needs a free Google account.
The full text of the lab is on the site: https://corso-rag.giuseppebosi.com/en/moduli/03-testo-cercabile/LAB/

In [ ]:
# Prepara l'ambiente / Set up the environment (about 1 minute)
import os, sys, subprocess
if sys.version_info[:2] not in [(3, 11), (3, 12)]:
    print("ATTENZIONE: questo corso e' provato con Python 3.11 o 3.12; hai "
          f"{sys.version_info.major}.{sys.version_info.minor}. "
          "WARNING: the course is tested on Python 3.11 or 3.12.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE, "rag-chatbot-course")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/JoeBosi/rag-chatbot-course.git", REPO_DIR], check=True)
os.chdir(os.path.join(REPO_DIR, "corso"))
!pip install -q openai python-dotenv httpx PyYAML pypdf fastembed numpy transformers
print("Cartella di lavoro / working folder:", os.getcwd())

Questo laboratorio **non chiama mai il modello**: non serve nessuna chiave. La prima esecuzione della ricerca semantica scarica circa 220 MB di modello (un paio di minuti, una volta sola).

---

This lab **never calls the model**: no key needed. The first run of semantic search downloads about 220 MB of model (a couple of minutes, once).

## Starter o soluzione? / Starter or solution?

Nel laboratorio scrivi tu alcune funzioni (i TODO). Qui sotto scegli:
`USA_SOLUZIONE = True` copia la soluzione, cosi' tutte le celle girano subito;
`USA_SOLUZIONE = False` copia lo **starter** con i TODO: apri il file nel pannello **File** a sinistra
(cartella `rag-chatbot-course/corso/memoria`), scrivi la funzione, salva e rilancia le celle.

---

In the lab you write some functions yourself (the TODOs). Choose below:
`USA_SOLUZIONE = True` copies the solution so every cell runs straight away;
`USA_SOLUZIONE = False` copies the **starter** with the TODOs: open the file in the **Files** panel on
the left (folder `rag-chatbot-course/corso/memoria`), write the function, save, and run the cells again.

In [ ]:
USA_SOLUZIONE = True   # False = parti dallo starter e scrivi tu i TODO
MOD = "moduli/03-testo-cercabile"
ORIGINE = "soluzione" if USA_SOLUZIONE else "starter"
!mkdir -p memoria
!cp $MOD/$ORIGINE/pezzi.py memoria/
!cp $MOD/$ORIGINE/lessicale.py memoria/
!cp $MOD/$ORIGINE/semantica.py memoria/
!cp $MOD/$ORIGINE/ibrida.py memoria/

## Passo 1: spezzare

Deve dare 6 pezzi alla cieca, 8 per titoli, e `48 pezzi da 12 documenti`.

---

## Step 1: splitting

It must give 6 chunks blind, 8 by headings, and `48 pezzi da 12 documenti`.

In [ ]:
!python memoria/pezzi.py progetto-finale/corpus/manuali/termovex-tx24.md --cieco
!python memoria/pezzi.py progetto-finale/corpus/manuali/termovex-tx24.md
!python memoria/pezzi.py

## Passo 2: BM25

Il primo risultato deve venire da `manuali/greencut-rm46.md`. L'idf di `bpr6es` deve essere molto piu' grande di quello di `il`.

---

## Step 2: BM25

The first result must come from `manuali/greencut-rm46.md`. The idf of `bpr6es` must be much larger than that of `il`.

In [ ]:
import sys; sys.path.insert(0, "memoria")
from pezzi import carica_pezzi
from lessicale import Bm25
motore = Bm25(carica_pezzi())
for p in motore.cerca("Con che coppia si stringe il bullone della lama?"):
    print(round(p["punteggio"], 2), p["id"])
print("idf bpr6es:", round(motore.idf("bpr6es"), 3), " idf il:", round(motore.idf("il"), 3))

## Passo 3: embedding e coseno

Le due frasi senza parole in comune devono risultare molto piu' simili fra loro che alla terza.

---

## Step 3: embeddings and cosine

The two sentences with no words in common must be far more similar to each other than to the third.

In [ ]:
from semantica import vettori, similitudine_coseno
v = vettori(["il rasaerba non parte", "il tagliaerba non si avvia", "la caldaia va in errore E4"])
print("rasaerba vs tagliaerba:", round(float(similitudine_coseno(v[0], v[1:2])[0]), 3))
print("rasaerba vs caldaia:   ", round(float(similitudine_coseno(v[0], v[2:3])[0]), 3))

In [ ]:
from semantica import Semantica
for p in Semantica(carica_pezzi()).cerca("Il tagliaerba non si accende"):
    print(round(p["punteggio"], 3), p["id"])

## Passo 5: misurare

Questa e' la tabella da ottenere esattamente:

```text
metodo        primi 1  primi 3  primi 5
BM25            4/10     8/10    10/10
semantica       5/10     7/10     8/10
ibrida          4/10     8/10    10/10
-- unione       7/10    10/10    10/10
```

---

## Step 5: measure

This is the table you must get exactly:

```text
metodo        primi 1  primi 3  primi 5
BM25            4/10     8/10    10/10
semantica       5/10     7/10     8/10
ibrida          4/10     8/10    10/10
-- unione       7/10    10/10    10/10
```

In [ ]:
!python moduli/03-testo-cercabile/starter/confronta.py

## Passo 7: verifica

---

## Step 7: check

In [ ]:
!python moduli/03-testo-cercabile/verifica.py